# Analytical FFA/FFT Sensitivity, Markov HMM Nulling & Heavy-Tail LLR Bounds

Demonstrates:
1. `pyloki.sensitivity.analytical`: FFA/EP vs incoherent FFT harmonic summing (`compare_ffa_vs_fft_flux_ratio`), folded $Z^2_m$, and $H$-test.
2. `pyloki.detection.scoring` + `pyloki.simulation.pulse`: 2-state Markov HMM forward-DP scorer (`markov_hmm_grid_score`, `viterbi_null_mask`, `markov_hmm_posterior_on`) for nulling/eclipsing pulsars.
3. `pyloki.sensitivity.heavy_tail`: Pareto ($eta > 2$) and lognormal Gaussian-convolved Neyman-Pearson LLR & KL divergence bounds.

In [1]:
import math

import numpy as np

from pyloki.detection import scoring
from pyloki.sensitivity import analytical, heavy_tail
from pyloki.simulation.pulse import generate_markov_nulling_envelope

# 1. Analytical FFA/EP vs incoherent 16-harmonic FFT flux ratio
for duty in (0.01, 0.02, 0.05, 0.10):
    comp = analytical.compare_ffa_vs_fft_flux_ratio(
        duty_cycle=duty,
        n_harmonics=16,
        fap=1e-7,
        target_pdet=0.5,
        rho_ep=0.92,
    )
    print(
        f"duty={duty:4.2f}: S_min(FFA)={comp.snr_min_ffa:5.2f}, "
        f"S_min(FFT16)={comp.snr_min_fft:5.2f}, "
        f"FFA/FFT={comp.flux_ratio_ffa_over_fft:.3f}"
    )

prof = analytical.gaussian_profile(0.03, n_grid=128)
sig_prof = 10.0 * prof / math.sqrt(128.0)
z2 = analytical.z2m_statistic(sig_prof, m=16)
h_val, best_m = analytical.h_test_statistic(sig_prof, m_max=20)
print(f"Folded Z^2_16 = {z2:.2f}, H-test = {h_val:.2f} (best_m={best_m})")

duty=0.01: S_min(FFA)= 5.65, S_min(FFT16)=13.12, FFA/FFT=0.431
duty=0.02: S_min(FFA)= 5.65, S_min(FFT16)=10.06, FFA/FFT=0.562
duty=0.05: S_min(FFA)= 5.65, S_min(FFT16)= 8.89, FFA/FFT=0.635
duty=0.10: S_min(FFA)= 5.65, S_min(FFT16)= 8.88, FFA/FFT=0.636
Folded Z^2_16 = 93.54, H-test = 39.81 (best_m=11)


In [2]:
# 2. 2-state Markov HMM scorer on an intermittent / nulling pulsar
rng = np.random.default_rng(202)
n_seg = 200
env = generate_markov_nulling_envelope(n_seg, p_on_off=0.05, p_off_on=0.015, rng=rng)
z_seg = rng.standard_normal(n_seg) + 2.5 * env

best_llr, amp, p10, p01 = scoring.markov_hmm_grid_score(z_seg)
on_mask = scoring.viterbi_null_mask(z_seg, amp, p10, p01)
p_on = scoring.markov_hmm_posterior_on(z_seg, amp, p10, p01)
snr_unweighted = float(np.sum(z_seg) / math.sqrt(n_seg))
snr_hmm_eq = math.sqrt(max(2.0 * best_llr, 0.0))
print(
    f"Nulling HMM: ON frac={float(np.mean(env)):.2f}, "
    f"Viterbi overlap={float(np.mean(on_mask == (env > 0.5))):.2f}, "
    f"mean P(ON|z)={float(np.mean(p_on)):.2f}, "
    f"unweighted S/N={snr_unweighted:.2f} vs HMM eq S/N={snr_hmm_eq:.2f}"
)

Nulling HMM: ON frac=0.13, Viterbi overlap=1.00, mean P(ON|z)=0.14, unweighted S/N=3.53 vs HMM eq S/N=11.00


In [3]:
# 3. Heavy-tail giant-pulse Neyman-Pearson LLR vs linear baseline
for dist, shape in (("pareto", 2.8), ("lognormal", 1.2)):
    model = heavy_tail.HeavyTailLikelihood(
        distribution=dist,
        mean_s=0.5,
        shape_param=shape,
        sigma_n=1.0,
    )
    print(
        f"Heavy-tail ({dist:9s}): D_KL(NP)={model.kl_divergence():.4f} "
        f"vs linear baseline={model.linear_kl_baseline():.4f}"
    )

Heavy-tail (pareto   ): D_KL(NP)=0.3069 vs linear baseline=0.1250
Heavy-tail (lognormal): D_KL(NP)=0.3331 vs linear baseline=0.1250
